In [1]:
import os
import pandas as pd
import gspread
import psycopg2
import boto3
from io import BytesIO
from dotenv import load_dotenv

In [2]:
load_dotenv()
db_name=os.getenv("DATABASE")
db_username=os.getenv("DB_USERNAME")
db_pass=os.getenv("DB_PASSWORD")
db_host=os.getenv("DB_HOST")


access_key_id=os.getenv("AWS_ACCESS_KEY_ID")
secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
s3_bucket_name=os.getenv("S3_BUCKET_NAME")
influencer_key=os.getenv("KEY1")
order_key=os.getenv("KEY2")
athena_db_name=os.getenv("ATHENA_DB")



sheet_name=os.getenv("SHEET_NAME")
tab_name=os.getenv("WORKSHEET_NAME")

In [3]:

def extraction():
    load_dotenv()
    db_name=os.getenv("DATABASE")
    db_username=os.getenv("DB_USERNAME")
    db_pass=os.getenv("DB_PASSWORD")
    db_host=os.getenv("DB_HOST")

    sheet_name=os.getenv("SHEET_NAME")
    tab_name=os.getenv("WORKSHEET_NAME")

    conn= psycopg2.connect(host=db_host, database=db_name, user=db_username, password=db_pass)
    pg_data=pd.read_sql_query("select * FROM historical.liffey_luxury_order_transactions",conn)
    conn.close()

    ##google pull
    my_credential=gspread.service_account(filename="datasets-lll-5a3914feb8ab.json")
    file_name=my_credential.open(sheet_name).worksheet(tab_name)
    get_data=file_name.get_all_records()
    
    ##dataframes
    google_data=pd.DataFrame(get_data)
    postgres_data=pd.DataFrame(pg_data)

    return google_data, postgres_data

In [4]:
google_data, postgres_data=extraction()

/tmp/ipykernel_6765/2574054498.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pg_data=pd.read_sql_query("select * FROM historical.liffey_luxury_order_transactions",conn)


In [5]:
##Transformation
def transformation(df1, df2):
    influencer_data, orders_data = df1, df2
    influencer_data['signup_date']=pd.to_datetime(influencer_data['signup_date'],errors='coerce')
    influencer_data['year']=influencer_data['signup_date'].dt.year.astype(str)
    influencer_data['month']=influencer_data['signup_date'].dt.strftime('%m')
    influencer_data=influencer_data.dropna(subset=['year','month'])

    influencer_data['influencer_code']=influencer_data['influencer_code'].astype(str).str.strip('@')

    return influencer_data, orders_data

In [6]:
influencer_data, orders_data=transformation(google_data,postgres_data)

In [7]:
def anthena_creation(df, tablename, s3_path):
    athena = boto3.client('athena')

    type_map = {
        'int64': 'BIGINT',
        'int32': 'INT',
        'float64': 'DOUBLE',
        'datetime64[ns]': 'TIMESTAMP',
        'datetime64[us]': 'TIMESTAMP',
        'bool': 'BOOLEAN',
        'object': 'STRING'
    }
    columns=", ".join([f"`{col}` {type_map.get(str(dtype).lower(), 'STRING')}" for col, dtype in df.dtypes.items()])

    create_table = f""" 
    CREATE EXTERNAL TABLE IF NOT EXISTS {athena_db_name}.{tablename} ({columns})
    STORED AS PARQUET
    LOCATION '{s3_path}'
    TBLPROPERTIES ('parquet.compress'='SNAPPY');
    """

    athena.start_query_execution(
        QueryString=create_table, QueryExecutionContext={'Database':athena_db_name},
        ResultConfiguration={'OutputLocation':os.getenv("Anthena_Output_S3")}
    )

    print(f"Anthena Table '{tablename} registration submitted")

In [8]:
def move_file_s3(df,key,tablename):

    load_dotenv()

    access_key_id=os.getenv("AWS_ACCESS_KEY_ID")
    secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
    s3_bucket_name=os.getenv("S3_BUCKET_NAME")
  

    s3=boto3.client('s3',aws_access_key_id=access_key_id, aws_secret_access_key=secret_access_key)

    pq_buffer=BytesIO()
    df.to_parquet(pq_buffer, engine='pyarrow',index=False, compression='snappy')

    s3.put_object(Bucket=s3_bucket_name, Key=key, Body=pq_buffer.getvalue() )

    s3_folder_path = f"s3://{s3_bucket_name}/{os.path.dirname(key)}/"

    anthena_creation(df, tablename, s3_folder_path)

    print(f"Full Load Complete: {key}")




In [9]:
move_file_s3(influencer_data, influencer_key, "influencer_data")
move_file_s3(orders_data, order_key, "order_data")

Anthena Table 'influencer_data registration submitted
Full Load Complete: lll/influencers_data/influencer_data.parquet
Anthena Table 'order_data registration submitted
Full Load Complete: lll/orders_data/orders_data.parquet
